# Telco Customer Churn — EDA Part 1: Data Quality

Exploratory analysis of the raw IBM Telco dataset, focused on structural integrity
and univariate distributions. We use `src/data/load.py` and `src/data/validate.py`
for data ingestion and schema validation, adding the visual layer here.

**Goal:** surface every data quality issue a production pipeline must handle
before any feature engineering touches the data.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Works whether notebook is run from repo root or notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import download_telco_data, load_raw
from src.data.validate import validate_dataframe
from src.utils.plotting import (
    CHURN_PALETTE,
    PALETTE,
    plot_bar,
    plot_histogram,
    save_fig,
    set_plot_style,
)

set_plot_style()
FIGURES_DIR = REPO_ROOT / "reports" / "figures"

In [ ]:
with open(REPO_ROOT / "configs" / "config.yaml") as fh:
    cfg = yaml.safe_load(fh)

RAW_PATH = REPO_ROOT / cfg["paths"]["raw_data"]
download_telco_data(RAW_PATH, urls=cfg["data"]["download_urls"])
df = load_raw(RAW_PATH)

print(f"Loaded  {len(df):,} rows \u00d7 {df.shape[1]} columns")
print(f"Memory  {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

## 1. Shape and Schema

The 21 columns fall into four groups:

| Group | Columns |
|---|---|
| **Identifier** | `customerID` |
| **Numeric** | `tenure`, `MonthlyCharges`, `TotalCharges` (read as str — see §3) |
| **Binary categorical** | gender, Partner, Dependents, PhoneService, PaperlessBilling, Churn |
| **Multi-value categorical** | InternetService, Contract, PaymentMethod, 6 service-addon columns |

`SeniorCitizen` is stored as an integer (0/1) rather than the Yes/No convention
used by every other binary column — a schema inconsistency we document but do not
treat as an error.

In [ ]:
dtype_summary = pd.DataFrame({
    "dtype":      df.dtypes,
    "nunique":    df.nunique(),
    "null_count": df.isna().sum(),
    "sample":     df.iloc[0],
})
dtype_summary

## 2. Schema Validation

`validate_dataframe()` checks column presence, dtype expectations, null constraints
for non-nullable fields, categorical allowed-value sets, and duplicate customerIDs.
Errors would indicate data that will break downstream code; warnings flag known
quirks handled explicitly in preprocessing.

In [ ]:
result = validate_dataframe(df)
print(result)
print(f"\nPasses validation: {result.is_valid}")

## 3. Duplicates and Missingness

No pandas NaN values exist in the raw frame. However, `TotalCharges` hides
an **encoded-missing pattern**: the source CSV stores blank strings for customers
with `tenure == 0`, who have not yet been billed a full month.

This is not a data error — it is a source-system design choice. The preprocessing
step that coerces `TotalCharges` to float will produce exactly 11 NaN values,
which must then be imputed (zero or `tenure × MonthlyCharges`).

In [ ]:
n_dupes = int(df["customerID"].duplicated().sum())
n_nan_total = int(df.isna().sum().sum())

blank_tc_mask = df["TotalCharges"].str.strip() == ""
n_blank = int(blank_tc_mask.sum())

print(f"Duplicate customerIDs       : {n_dupes}")
print(f"Pandas NaN (entire frame)   : {n_nan_total}")
print(f"TotalCharges blank strings  : {n_blank}")
print()
df.loc[blank_tc_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

In [ ]:
tc_numeric = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: tenure vs TotalCharges, highlighting the blank-string rows
axes[0].scatter(
    df.loc[~blank_tc_mask, "tenure"],
    tc_numeric[~blank_tc_mask],
    alpha=0.25, s=7, color=PALETTE[0], label="Normal",
)
axes[0].scatter(
    df.loc[blank_tc_mask, "tenure"],
    np.zeros(n_blank),
    color=PALETTE[1], s=60, zorder=5,
    label=f"Blank TotalCharges (n={n_blank})",
)
axes[0].set_xlabel("tenure (months)")
axes[0].set_ylabel("TotalCharges (parsed to float)")
axes[0].set_title("TotalCharges vs Tenure")
axes[0].legend(frameon=False, fontsize=9)

# Bar: breakdown of blank-TC rows by tenure group
breakdown = pd.Series({
    "tenure=0, blank TC":  int((blank_tc_mask & (df["tenure"] == 0)).sum()),
    "tenure=0, has TC":    int((~blank_tc_mask & (df["tenure"] == 0)).sum()),
    "tenure>0, blank TC":  int((blank_tc_mask & (df["tenure"] > 0)).sum()),
})
plot_bar(
    breakdown, ax=axes[1],
    color=[PALETTE[1], PALETTE[0], PALETTE[2]],
    title="Blank TotalCharges by tenure group",
    pct=False,
)

fig.suptitle("TotalCharges data quality", fontweight="bold")
fig.tight_layout()
save_fig(fig, "03_total_charges_quirk", FIGURES_DIR)
plt.show()
print("Saved: 03_total_charges_quirk.png")

## 4. Target Variable Distribution

Churn rate is ~26.5%, creating a **moderate class imbalance**. A naive classifier
that always predicts "No" would achieve ~73.5% accuracy — meaningless for a
retention use case. Later we will tune the classification threshold using a
business cost matrix (false-negative cost = lost customer lifetime value;
false-positive cost = wasted retention offer).

In [ ]:
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
churn_rate = churn_counts["Yes"] / len(df) * 100
print(f"Churn rate: {churn_rate:.1f}%  ({churn_counts['Yes']:,} / {len(df):,} customers)")

fig, ax = plt.subplots(figsize=(5, 4))
plot_bar(
    churn_counts, ax=ax,
    color=[CHURN_PALETTE[k] for k in churn_counts.index],
    title=f"Churn distribution (rate = {churn_rate:.1f}%)",
)
ax.set_xlabel("Churn")
fig.tight_layout()
save_fig(fig, "01_churn_rate", FIGURES_DIR)
plt.show()
print("Saved: 01_churn_rate.png")

## 5. Numeric Feature Distributions

- **tenure** (0–72 months): Bimodal — large spikes at month 1 (new customers) and
  month 70–72 (loyal customers). Very few mid-tenure customers, suggesting either
  early attrition or gradual ramp-up from a recent product launch.
- **MonthlyCharges** ($18–$119): Right-skewed with a mode around $20 (basic phone-
  only plans) and a secondary cluster near $80 (premium fibre + add-ons).
- **TotalCharges**: A product of tenure and monthly charges; inherits the bimodal
  tenure pattern, stretched by charges variation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_histogram(df["tenure"], ax=axes[0], bins=36,
               color=PALETTE[0], title="tenure (months)")
plot_histogram(df["MonthlyCharges"], ax=axes[1], bins=40,
               color=PALETTE[3], title="MonthlyCharges ($)")
plot_histogram(tc_numeric.rename("TotalCharges"), ax=axes[2], bins=40,
               color=PALETTE[4], title="TotalCharges ($ parsed)")

fig.suptitle("Numeric feature distributions", fontweight="bold")
fig.tight_layout()
save_fig(fig, "02_numeric_distributions", FIGURES_DIR)
plt.show()
print("Saved: 02_numeric_distributions.png")

## 6. Contract Type

Contract type is widely considered the strongest single predictor of churn.
Month-to-month subscribers (~55% of customers) face zero switching costs;
two-year contracts are associated with very low churn. This imbalance in
contract mix will strongly influence the feature importance rankings we
will see from SHAP later.

In [ ]:
contract_counts = df["Contract"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
plot_bar(
    contract_counts, ax=ax,
    color=PALETTE[:3], horizontal=True,
    title="Contract type distribution",
)
fig.tight_layout()
save_fig(fig, "04_contract_type", FIGURES_DIR)
plt.show()
print("Saved: 04_contract_type.png")

## 7. Internet Service and Payment Method

**Internet service** splits roughly 44% DSL / 34% Fibre optic / 22% no internet.
Fibre optic customers tend to pay higher monthly charges and have shown higher
churn rates in prior analyses of this dataset — worth examining in the bivariate EDA.

**Payment method** shows ~34% of customers on electronic check — a cohort often
associated with month-to-month contracts and higher churn risk.

In [ ]:
internet_counts = df["InternetService"].value_counts()
payment_counts = df["PaymentMethod"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_bar(internet_counts, ax=axes[0], color=PALETTE[:3],
         title="Internet service type")
plot_bar(payment_counts, ax=axes[1], color=PALETTE[:4],
         horizontal=True, title="Payment method")

fig.tight_layout()
save_fig(fig, "05_service_profile", FIGURES_DIR)
plt.show()
print("Saved: 05_service_profile.png")

## Data Quality Summary

| Finding | Detail | Action Required |
|---|---|---|
| **Shape** | 7,043 rows × 21 columns | None |
| **Duplicate IDs** | Zero | None |
| **Pandas NaN** | Zero across all columns | None |
| **TotalCharges blank strings** | 11 rows; all have `tenure == 0` | Coerce to float → 11 NaN; impute with `0` or `tenure × MonthlyCharges` |
| **TotalCharges dtype** | Read as `str` to preserve blanks | Parse to `float64` in preprocessing |
| **SeniorCitizen encoding** | Stored as `int` (0/1), not `Yes/No` | Documented in Pydantic schema; treat as binary numeric or convert |
| **Class imbalance** | 26.5% churn rate | Cost-sensitive threshold optimisation; avoid raw accuracy as metric |
| **Bimodal tenure** | Peaks at month 1 and months 60–72 | Consider `tenure_bucket` feature |
| **Dual MonthlyCharges peaks** | ~$20 basic plans, ~$80 fibre/premium | Consider log transform or service-tier feature |

**Bottom line:** The raw dataset is structurally sound — no missing schema columns,
no duplicate IDs, all categoricals within their allowed value sets. The only
mandatory preprocessing step is coercing `TotalCharges` from `str` to `float`.
Everything else in this table is a modelling consideration, not a data error.